# P1-L2｜收益—风险账本

> **对应课程**：第 5–8 课  
> **模块**：P1-M2 · 收益、时间价值与风险  
> **数据身份**：固定随机种子模拟教学数据，不是真实证券行情。

用统一时点和频率核算收益、复利、波动、相关性与组合风险。

本实验向最终策略报告提供：**收益、风险与基准口径**。

In [1]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "fixtures").exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data" / "fixtures"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RANDOM_SEED = 365
print("项目:", ROOT)
print("数据:", DATA_DIR)
print("数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情")


项目: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations
数据: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations/data/fixtures
数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情


## 建立收益—风险账本

账本同时保存资产收益、基准、组合权重、成本口径和风险指标，避免只展示一个年化数字。

In [2]:
from src.quant_core import annualized_return, annualized_volatility, max_drawdown

daily = pd.read_csv(DATA_DIR / "daily_prices.csv", parse_dates=["date"])
close = daily.pivot(index="date", columns="symbol", values="adj_close").sort_index()
returns = close.pct_change(fill_method=None).dropna()

assets = ["SIM_VALUE", "SIM_GROWTH", "SIM_DEFENSIVE"]
weights = pd.Series([0.35, 0.35, 0.30], index=assets)
portfolio = returns[assets].mul(weights, axis="columns").sum(axis=1)
covariance = returns[assets].cov()
variance_matrix = float(weights @ covariance @ weights)

ledger = pd.Series({
    "annual_return": annualized_return(portfolio),
    "annual_volatility": annualized_volatility(portfolio),
    "max_drawdown": max_drawdown(portfolio),
    "daily_variance_from_matrix": variance_matrix,
    "daily_variance_from_series": portfolio.var(ddof=1),
})
np.testing.assert_allclose(ledger["daily_variance_from_matrix"], ledger["daily_variance_from_series"])
display(ledger.to_frame("value"))

,value
annual_return,0.001013
annual_volatility,0.135741
max_drawdown,-0.193092
daily_variance_from_matrix,0.000073
daily_variance_from_series,0.000073


## 验收

- 关键计算能由小例子或独立公式核对；
- 数据时点、信号时点和成交时点没有混用；
- 结果同时包含金融含义和失败边界；
- 产物能够进入第 29 课策略报告和第 30 课能力矩阵。